In [1]:
import pandas as pd
import gzip
import json

In [2]:
print(f"🔍 Reading gzip NDJSON file: {'/m/cs/scratch/sinitaivas/bluesky_firehose/firehose_stream/2024-12-17/2024-12-17T21.ndjson.gz'}")
records = []
with gzip.open('/m/cs/scratch/sinitaivas/bluesky_firehose/firehose_stream/2024-12-17/2024-12-17T21.ndjson.gz', "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue  # skip any malformed line
        if i >= 50000:  # optional: read only first N lines for faster testing
            break

if not records:
    raise ValueError("No valid JSON records found in the NDJSON file!")

df = pd.DataFrame(records)

🔍 Reading gzip NDJSON file: /m/cs/scratch/sinitaivas/bluesky_firehose/firehose_stream/2024-12-17/2024-12-17T21.ndjson.gz


In [3]:
df.columns.to_list()

['seq',
 'collected_at',
 'commit_time',
 'action',
 'type',
 'uri',
 'author',
 'cid',
 'createdAt',
 'subject',
 '$type',
 'text',
 'embed',
 'facets',
 'langs',
 'reply',
 'via',
 'avatar',
 'description',
 'displayName',
 'labels',
 'banner',
 'pinnedPost',
 'list',
 'tags',
 'name',
 'purpose',
 'descriptionFacets',
 'feeds',
 'updatedAt',
 'post',
 'allow',
 'hiddenReplies',
 'detachedEmbeddingUris',
 'embeddingRules',
 'allowIncoming',
 'bridgyOriginalUrl',
 'bridgyOriginalText',
 'did']

In [5]:
df.to_csv("sample.csv", index = None)

In [8]:

# Define mapping of action types
action_map = {
    "app.bsky.feed.post": "posts",
    "app.bsky.feed.repost": "reposts",
    "app.bsky.graph.follow": "follows",
    "app.bsky.feed.like": "likes"
}

# Filter only known types
df = df[df["type"].isin(action_map.keys())]

# Map each action to a category (post, repost, follow, like)
df["action_category"] = df["type"].map(action_map)
summary = (
df.groupby(["author", "action_category"])
.size()
.unstack(fill_value=0)
.reset_index()
)
summary

/tmp/ipykernel_94491/463707551.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["action_category"] = df["type"].map(action_map)


action_category,author,follows,likes,posts,reposts
0,did:plc:223f5neubhxazapvd5mmwioi,0,2,0,0
1,did:plc:226w3kve33nnfblfixvwwgto,2,0,0,0
2,did:plc:227mqd3o26tynduwdgdbxqu6,1,0,0,0
3,did:plc:22coqmaear2fwlrollu5hoax,1,2,1,0
4,did:plc:22gvahvnw2gk3eru5ildy4dj,0,1,0,0
...,...,...,...,...,...
19926,did:plc:zzozknebkvtval5uurylqv2n,0,2,2,0
19927,did:plc:zzphfrojjsktrshfiwpj3daa,0,1,0,0
19928,did:plc:zzsvmwqu6ec4dq23m2nrgj2u,0,0,0,1
19929,did:plc:zzugejr3gecohv7day6xwxbi,0,1,0,0
